# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [1]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc  
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path, preprocessed_path_freezed
from missing_data import compute_availability_metrics

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")        
today_day = pd.Timestamp.today().normalize()       
today_str = "01092025"
# --- Path -------------------------------------------------------------

datapath = Path(raw_path) / f"export_tiki_{today_str}"  

In [2]:
datapath

PosixPath('/sc-projects/sc-proj-cc15-preact/SP6/raw/export_tiki_01092025')

## 1. Passive Data

### 1.1 Load most recent passive data

In [3]:
# actual passive + ema_data
file_pattern = os.path.join(datapath, "epoch_part*.csv")
file_list = glob.glob(file_pattern)
file_list.sort()
df_complete = pd.concat((pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list), ignore_index=True)

In [4]:
# Extract customer identifier
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp"]:
    df_complete[col] = (
        pd.to_datetime(df_complete[col], utc=True, errors="coerce", unit="ms")
    )


In [5]:
df_complete = df_complete[['customer', 'startTimestamp', 'endTimestamp', 'timezoneOffset', 'type',
       'stringValue', 'booleanValue', 'doubleValue', 'longValue']]

### 1.2 Load big backup dataset

In [6]:
# Merge with backup data
backup_path = preprocessed_path + "/backup_passive_05052025.parquet"
# backup_path = preprocessed_path + "/backup_passive_last.feather"
df_backup = df_backup = pd.read_parquet(backup_path)

In [7]:

latest_timestamp = df_backup['startTimestamp'].max()
df_complete_filtered = df_complete[df_complete['startTimestamp'] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [8]:
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

In [9]:
df_backup_recent['float_value'] = df_backup_recent['doubleValue'].fillna(df_backup_recent['longValue'])


In [10]:
#df_backup_recent = df_backup_recent.drop(columns=['doubleValue', 'longValue', 'stringValue'])


### 1.4 Create additional columns

In [11]:

# Duration in seconds
df_backup_recent['start_end'] = (
    df_backup_recent['endTimestamp'] - df_backup_recent['startTimestamp']
).dt.total_seconds()

# Extract date and hour
df_backup_recent['startTimestamp_day']  = df_backup_recent['startTimestamp'].dt.normalize()
df_backup_recent['startTimestamp_hour'] = df_backup_recent['startTimestamp'].dt.hour


## 2. Monitoring data

In [12]:
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [13]:
df_monitoring = df_monitoring.copy()
df_monitoring.rename(columns = {"Pseudonym": "customer", "EMA_ID": "ema_id", "Status": "status",
                                "Studienversion":"study_version", "FOR_ID":"for_id", 
                           "Start EMA Baseline": "ema_base_start", "Ende EMA Baseline": "ema_base_end", 
                           "Freischaltung/ Start EMA T20": "ema_t20_start","Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", "T20=Post":"t20_post" }, inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'customer', 'study_version', 'status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

df_monitoring["customer"] = df_monitoring["customer"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)

### 2.1 Merge relevant columns with passive data

In [14]:
df_monitoring_short = df_monitoring[["customer", "for_id","study_version", "ema_base_start"]]

In [15]:
df_complete_filtered= df_complete_filtered.merge(df_monitoring_short, on="customer", how="right")

In [16]:
object_cols = ["booleanValue", "stringValue","customer", "type", "study_version"] 

# Fill NaN values with -99 for the specified columns
for col in object_cols:
    df_complete_filtered[col] = df_complete_filtered[col].fillna(-99)

# Convert "booleanValue" to boolean
df_complete_filtered['booleanValue'] = df_complete_filtered['booleanValue'].apply(lambda x: bool(x) if x != -99 else False)

# Convert "stringValue", "status", "study_version" to string using StringDtype
df_complete_filtered['stringValue'] = df_complete_filtered['stringValue'].astype('string')
df_complete_filtered['study_version'] = df_complete_filtered['study_version'].astype('string')
df_complete_filtered['customer'] = df_complete_filtered['customer'].astype('string')
df_complete_filtered['type'] = df_complete_filtered['type'].astype('string')


In [17]:
data_type_groups = {
    'GPS': ["Latitude"],
    # Add more groups as needed
    'Activity': ["Steps"],
    'Sleep': ["SleepBinary"],
    'Heart_Rate': ["HeartRate"]
    # Add more groups as needed
}

## EMA data

#### 1. Load and match relevant data from separate .csv files

In [18]:
df_backup_recent= df_backup_recent.merge(df_monitoring_short, on="customer", how="right")

In [19]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = ['customer','for_id', 'type', 'startTimestamp','endTimestamp','start_end','doubleValue', 'longValue', 'booleanValue','startTimestamp_day', 
    'startTimestamp_hour','ema_base_start', 'study_version']

df_passive_final = df_backup_recent[keep_cols_passive]

## 3. EMA data

#### 3.1 Load and match relevant data from separate .csv files

In [20]:

# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session        = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers        = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice         = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions      = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire  = pd.read_csv(datapath / "questionnaires.csv", low_memory=False)  # ohne Komma!


In [21]:
session

,id,user,study,questionnaire,sessionRun,expirationTimestamp,createdAt,completedAt
0,6954,APbNFdORpGGj7vpb,25,56,0,1682583300000,1682582801477,1.682583e+12
1,6963,APbNFdORpGGj7vpb,25,68,0,1682590500000,1682589815644,1.682590e+12
2,7004,APbNFdORpGGj7vpb,25,76,0,1682684100000,1682683238257,1.682683e+12
3,7010,APbNFdORpGGj7vpb,25,84,0,1682691300000,1682690465941,1.682691e+12
4,7012,APbNFdORpGGj7vpb,25,90,0,1682694900000,1682694098187,1.682694e+12
...,...,...,...,...,...,...,...,...
46011,153580,p1zHtcdMMBiyyMWA,39,120,1,1756669140000,1756667557973,1.756668e+12
46012,153581,SZPebtjVWkkDXppo,33,104,3,1756669320000,1756667559242,1.756668e+12
46013,153584,0AlyaEtHo8K3Cyo8,38,102,0,1756670400000,1756668750390,1.756669e+12
46014,153585,mASGckHiLqgF7A5d,38,102,3,1756670400000,1756668910402,1.756669e+12


In [22]:
# session data
session["user"] = session["user"].str[:4]
session.rename(columns = {"user":"customer","completedAt": "quest_complete", "createdAt": "quest_create", "expirationTimestamp": "quest_expir"}, inplace=True)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["quest_create", "quest_complete", "quest_expir"]:
    session[col] = (
        pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")
    )

df_sess = session[["customer", "sessionRun", "quest_create", "quest_complete", "study", "quest_expir"]]

In [23]:
answers["user"] = answers["user"].str[:4]
answers = answers[["user", "questionnaire", "study", "question", "element",
                   "createdAt", "order", "questionnaireSession"]]

answers["createdAt"] = (
    pd.to_datetime(answers["createdAt"], unit="ms", utc=True, errors="coerce")
)

answers.rename(columns={"user": "customer", "createdAt": "quest_create"}, inplace=True)

In [24]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text":"choice_text"}, inplace=True)

In [25]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id":"question","title":"quest_title"}, inplace=True)

In [26]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(columns={"id":"questionnaire","name":"questionnaire_name"}, inplace=True)

In [27]:
answer_merged = pd.merge(answers, choice, on= ["question","element"])
answer_merged = pd.merge(answer_merged, questions, on= "question")
answer_merged = pd.merge(answer_merged, questionnaire, on= "questionnaire")
answer_merged["quest_create_day"] = answer_merged.quest_create.dt.normalize()

In [28]:
answer_merged = pd.merge(answer_merged, df_monitoring, on = "customer")

#### 3.2 Calculate EMA coverage

In [29]:
df_sess = pd.merge(df_sess, df_monitoring, on = "customer")

In [30]:
df_sess = df_sess[['customer', 'sessionRun', 'quest_create', 'quest_complete', 'study',
       'for_id', 'ema_id', 'study_version', 'status', 't20_post',
       'ema_base_start', 'ema_base_end','ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end']]

In [31]:
df_sess = df_sess.copy()
df_sess["quest_create_day"] = df_sess.quest_create.dt.normalize()
df_sess["quest_complete_day"] = df_sess.quest_complete.dt.normalize()

df_sess["quest_create_hour"] = df_sess.quest_create.dt.hour
df_sess["quest_complete_hour"] = df_sess.quest_complete.dt.hour

In [32]:
# count number of completed EMA beeps in first phase
df_sess1 = df_sess.loc[df_sess.study.isin([24,25])]
df_sess1 = df_sess1.copy()

df_sess2 = df_sess.loc[df_sess.study.isin([33,34])]
df_sess2 = df_sess2.copy()

df_sess3 = df_sess.loc[df_sess.study.isin([38,39])]
df_sess3 = df_sess3.copy()

In [33]:
df_sess1['quest_complete_relative1'] = (df_sess1['quest_complete_day'] - df_sess1['ema_base_start']).dt.days


sess_count1 = df_sess1.dropna(subset=["quest_create"]).groupby("customer")["quest_create"].size()\
.reset_index()
sess_count1 = sess_count1.rename(columns = {"quest_create":"nquest_EMA1"})

# count number of completed EMA beeps in second phase
sess_count2 = df_sess2.dropna(subset=["quest_create"]).groupby("customer")["quest_create"].size()\
.reset_index()
sess_count2 = sess_count2.rename(columns = {"quest_create":"nquest_EMA2"})

# count number of completed EMA beeps in second phase
sess_count3 = df_sess3.dropna(subset=["quest_create"]).groupby("customer")["quest_create"].size()\
.reset_index()
sess_count3 = sess_count3.rename(columns = {"quest_create":"nquest_EMA3"})

In [34]:
df_sess = df_sess.merge(sess_count1, on=['customer'], how='left')
df_sess = df_sess.merge(sess_count2, on=['customer'], how='left')
df_sess = df_sess.merge(sess_count3, on=['customer'], how='left')

#### 3.3 Calculate auxiliary variables

In [35]:
df_ema_content = answer_merged.copy()

In [36]:
# 1. Date and Time Manipulations
df_ema_content['weekday'] = df_ema_content['quest_create'].dt.day_name()
df_ema_content['createdAt_day'] = df_ema_content['quest_create'].dt.floor('D')

date_cols = ['ema_base_start', 'ema_t20_start', 'ema_post_start']
for col in date_cols:
    df_ema_content[col] = pd.to_datetime(df_ema_content[col], dayfirst=True, errors='coerce')

# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Winter'

df_ema_content['season'] = df_ema_content['quest_create'].dt.month.apply(get_season)

# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return 'Early Morning'
    elif 8 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df_ema_content['time_of_day'] = df_ema_content['quest_create'].dt.hour.apply(get_time_of_day)

# 2. Study Mapping / String Manipulation
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
df_ema_content['assess'] = df_ema_content['study'].map(study_mapping)
df_ema_content['quest_title'] = df_ema_content['quest_title'].str.replace('_morning', '', regex=False)

# 3. Weekend Indicator
df_ema_content['weekend'] = df_ema_content['weekday'].isin(['Saturday', 'Sunday']).astype(int)

# 4. Questionnaire Number
df_ema_content['quest_nr'] = df_ema_content['questionnaire_name'].str.extract(r'(\d+)').astype(float)

# 5. Count unique questionnaires per day
df_ema_content['n_quest'] = (
    df_ema_content.groupby(['study', 'customer', 'createdAt_day'])['questionnaire_name']
                  .transform('nunique')
)

# 6. Unique Day Identifier
df_ema_content['quest_nr_str'] = df_ema_content['quest_nr'].fillna('unknown').astype(str)
df_ema_content['unique_day_id'] = (
    df_ema_content['createdAt_day'].dt.strftime('%Y%m%d') + '_' + df_ema_content['quest_nr_str']
)

# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content['ema_relative_start'] = (
    df_ema_content.groupby(['customer', 'assess'])['createdAt_day'].transform('min')
)
df_ema_content['ema_relative_end'] = (
    df_ema_content.groupby(['customer', 'assess'])['createdAt_day'].transform('max')
)

# 8. Absolute & Relative Day Index
df_ema_content['absolute_day_index'] = (
    df_ema_content['createdAt_day'] - df_ema_content['ema_relative_start']
).dt.days + 1

df_ema_content['relative_day_index'] = (
    df_ema_content.groupby(['customer', 'assess'])['createdAt_day']
                  .rank(method='dense').astype(int)
)

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
df_ema_content = df_ema_content[df_ema_content['absolute_day_index'] <= max_allowed_days].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content['absolute_day_index'] > max_allowed_days]
if not high_indices.empty:
    print("Warning: High absolute_day_index vorhanden:", high_indices['customer'].unique())
else:
    print("All entries have absolute_day_index <= 16.")

# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(subset=['customer', 'assess', 'unique_day_id']).copy()
df_unique['questionnaire_counter'] = df_unique.groupby(['customer', 'assess']).cumcount() + 1
df_ema_content = df_ema_content.merge(
    df_unique[['customer', 'assess', 'unique_day_id', 'questionnaire_counter']],
    on=['customer', 'assess', 'unique_day_id'],
    how='left'
)

# 12. Missing Data behandeln
df_ema_content['assess'] = df_ema_content['assess'].fillna('unknown')
df_ema_content['absolute_day_index'] = df_ema_content['absolute_day_index'].where(
    df_ema_content['ema_relative_start'].notna(), np.nan
)

# 13. Falsche Person entfernen
filter_criteria = (
    (df_ema_content['customer'] == 'UfMn') &
    (df_ema_content['study'] == 25) &
    (df_ema_content['quest_create'] > '2024-02-08')
)
df_ema_content = df_ema_content[~filter_criteria]
# **Optional: View the Updated DataFrame**
# print(df_ema_content.head())


All entries have absolute_day_index <= 16.


In [37]:
filter_criteria = (df_ema_content['customer'] == 'UfMn') & \
                  (df_ema_content['study'] == 25) & \
                  (df_ema_content['quest_create'] > '2024-02-08')

# Drop the entries that match the criteria of the wrong individual
df_ema_content = df_ema_content[~filter_criteria]

In [38]:
df_ema_base = df_ema_content[["customer", 'ema_relative_start', 'ema_relative_end']]
df_ema_base = df_ema_base.drop_duplicates()

In [39]:
df_complete_ema = df_complete_filtered.merge(df_ema_base, on = "customer", how="left")

In [40]:
df_complete_ema_final = pd.concat([df_backup, df_complete_ema], ignore_index=True)

In [41]:
df_complete_ema_final['customer'] = df_complete_ema_final['customer'].astype('category')
df_complete_ema_final['type'] = df_complete_ema_final['type'].astype('category')

# Downcast numeric columns
df_complete_ema_final['doubleValue'] = pd.to_numeric(df_complete_ema_final['doubleValue'], errors='coerce', downcast='float')
df_complete_ema_final['longValue'] = pd.to_numeric(df_complete_ema_final['longValue'], errors='coerce', downcast='integer')

### Export passive, EMA and Monitoring

In [42]:
backup_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_path)

preprocessed_path_final = preprocessed_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

#preprocessed_path_freezed_final = preprocessed_path_freezed + "/code_check" + "/backup_passive_recent.parquet"
#df_passive_final.to_parquet(preprocessed_path_freezed_final)

with open(preprocessed_path + '/ema_adherence_data.pkl', 'wb') as file:
    pickle.dump(df_sess, file)

    
with open(preprocessed_path + '/monitoring_data.pkl', 'wb') as file:
    pickle.dump(df_monitoring, file)

    
with open(preprocessed_path + '/ema_content.pkl', 'wb') as file:
    pickle.dump(df_ema_content, file)

ArrowInvalid: ('Could not convert 1.0 with type float: tried to convert to boolean', 'Conversion failed for column booleanValue with type object')

In [44]:
df_passive_final.startTimestamp.max()

Timestamp('2025-09-01 04:01:02+0000', tz='UTC')

In [ ]:

# Export df_adherence as CSV
df_sess_csv_path = preprocessed_path + '/ema_adherence_data.csv'
df_sess.to_csv(df_sess_csv_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path + '/monitoring_data.csv'
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path + '/ema_content.csv'
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
#df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
#df_ema_content.to_csv(df_ema_content_csv_path, index=False)



## Redcap data (optional, cause not complete yet)

In [ ]:
df_redcap = pd.read_csv(redcap_path + "FOR5187_DATA_2025-01-07_1511.csv", low_memory=False)
df_redcap_zert = pd.read_csv(redcap_path + "ZERTIFIZIERUNGFOR518_DATA_2025-01-07_1518.csv", low_memory=False)

In [ ]:
df_redcap_zert = df_redcap_zert[['for_id', 'redcap_event_name',
       'basic_documentation_sheet_timestamp',  'age', 'gender','scid_cv_prim_cat',
       'marital_status', 'partnership', 'graduation', 'profession', 'ema_start_date',
       'years_of_education', 'employability', 'ses', 'ema_smartphone', 'ema_sleep', 'ema_watch', 'prior_treatment', 'ema_special_event', 'psychotropic', 'somatic_problems']]

In [ ]:
df_redcap = df_redcap[['for_id', 'redcap_event_name',
       'basic_documentation_sheet_timestamp', 'age', 'gender', 'scid_cv_prim_cat',
       'marital_status', 'partnership', 'graduation', 'profession', 'ema_start_date', 
       'years_of_education', 'employability', 'ses', 'ema_smartphone', 'ema_sleep', 'ema_watch','prior_treatment', 'ema_special_event', 'psychotropic', 'somatic_problems']]

In [ ]:
df_redcap = pd.concat([df_redcap, df_redcap_zert],ignore_index=True)
#df_redcap = pd.merge(df_redcap, df_t20,on='for_id', suffixes=('_base', '_t20'), how="left")

In [ ]:
# Group by subject_id and merge rows
df_redcap_merged = (
    df_redcap
    .groupby('for_id', as_index=False)
    .agg({
        'ema_watch': 'max',  # Takes the non-null value
        **{col: 'first' for col in df_redcap.columns 
           if col not in ['for_id', 'ema_watch', 'redcap_event_name']}  # Keeps the first of other columns
    })
)

# Optionally drop the 'redcap_event_name' column
if 'redcap_event_name' in df_redcap_merged.columns:
    df_redcap_merged = df_redcap_merged.drop(columns=['redcap_event_name'])

In [ ]:
gender_mapping = {
    1: 'male',
    2: 'female',
    3: 'diverse',
    4: 'no gender',
    5: 'not specified'
}

scid_cv_cat_mapping = {
    1: 'Depressive Disorder',
    2: 'Specific Phobia',
    3: 'Social Anxiety Disorder',
    4: 'Agoraphobia and/or Panic Disorder',
    5: 'Generalized Anxiety Disorder',
    6: 'Obsessive-Compulsive Disorder',
    7: 'Post-Traumatic Stress Disorder'
}

marital_status_mapping = {
    1: 'single',
    2: 'married/registered partnership',
    3: 'divorced',
    4: 'separated',
    5: 'widowed',
    6: 'other'
}

employability_mapping = {
    0: 'employable',
    1: 'unemployable (on sick leave)',
    2: 'on disability pension',
    3: 'on retirement pension',
    4: 'other'
}

employability_mapping_simple = {
    0: 'yes',
    1: 'no',
    2: 'no',
    3: 'no',
    4: 'no'
}

graduation_mapping = {
    0: 'still in school',
    1: 'no school degree',
    2: 'elementary school degree or equivalent',
    3: 'middle school degree or equivalent',
    4: 'high school diploma/university entrance qualification',
    5: 'other'
}

profession_mapping = {
    0: 'still in training or studies',
    1: 'no training degree',
    2: 'vocational training, including technical school',
    3: 'university or college degree',
    4: 'other'
}

prior_treatment_mapping = {
    0: 'no prior treatment',
    1: 'outpatient psychotherapy',
    2: 'inpatient or partial inpatient treatment/psychotherapy',
    3: 'both',
    4: 'yes'
}

prior_treatment_mapping_simple = {
    0: 'no prior treatment',
    1: 'prior psychotherapy',
    2: 'prior inpatient',
    3: 'prior inpatient',
    4: 'prior psychotherapy'
}

psychotropic_medication_mapping = {
    0: 'no',
    1: 'yes'
}
somatic_mapping = {
    0: 'no',
    1: 'yes'
}
ema_smartphone_mapping = {
    1: 'iPhone',
    0: 'Android'
}

ema_special_event_mapping = {
    0: 'usual',
    1: 'special event'
}
def categorize_age(age):
    if 18 <= age <= 24:
        return 0
    elif 25 <= age <= 34:
        return 1
    elif 35 <= age <= 44:
        return 2
    elif 45 <= age <= 54:
        return 3
    elif 55 <= age <= 64:
        return 4
    else:
        return 5
    

In [ ]:
# Apply mappings
df_redcap_merged['gender_description'] = df_redcap_merged['gender'].map(gender_mapping)
df_redcap_merged['scid_cv_description'] = df_redcap_merged['scid_cv_prim_cat'].map(scid_cv_cat_mapping)
df_redcap_merged['marital_status_description'] = df_redcap_merged['marital_status'].map(marital_status_mapping)
df_redcap_merged['employability_description'] = df_redcap_merged['employability'].map(employability_mapping)
df_redcap_merged['employability_description_simple'] = df_redcap_merged['employability'].map(employability_mapping_simple)
df_redcap_merged['prior_treatment_description_simple'] = df_redcap_merged['prior_treatment'].map(prior_treatment_mapping_simple)
df_redcap_merged['graduation_description'] = df_redcap_merged['graduation'].map(graduation_mapping)
df_redcap_merged['profession_description'] = df_redcap_merged['profession'].map(profession_mapping)
df_redcap_merged['prior_treatment_description'] = df_redcap_merged['prior_treatment'].map(prior_treatment_mapping)
df_redcap_merged['ema_smartphone_description'] = df_redcap_merged['ema_smartphone'].map(ema_smartphone_mapping)
df_redcap_merged['ema_special_event_description'] = df_redcap_merged['ema_special_event'].map(ema_special_event_mapping)
df_redcap_merged['age_description'] = df_redcap_merged['age'].apply(categorize_age)
df_redcap_merged['somatic_description'] = df_redcap_merged['somatic_problems'].map(somatic_mapping)
df_redcap_merged['psychotropic_description'] = df_redcap_merged['psychotropic'].map(psychotropic_medication_mapping)



In [ ]:
# add FOR ID
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()
df_forid = df_monitoring[["for_id","customer"]]
df_redcap = pd.merge(df_forid, df_redcap_merged, on="for_id", how="left")

In [ ]:
valid_df = df_redcap.dropna(subset=['ema_start_date'])


In [ ]:
with open(preprocessed_path_freezed + f'/redcap_data.pkl', 'wb') as file:
    pickle.dump(valid_df, file)